# Analyze Eclipse Fault INC

In [ ]:
# Import Python libraries
!pip install -q -U kaleido==0.2.1 -q -U skimpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive (run once per session)
drive.mount("/content/drive")

# Data directory (adjust if you move the notebook)
source = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/FAULT/")
dest = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne/")

# Create directory if it does not exist
os.makedirs(dest, exist_ok=True)

print("Source directory:", source)
print("Output directory:", dest)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Source directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/FAULT
Output directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne


In [ ]:
from pathlib import Path

# All entries (files + subdirs)
for p in source.iterdir():
    print(p)

# Only files
for p in source.iterdir():
    if p.is_file():
        print(p)

These two files define fault geometries (FAULTS) and transmissibility multipliers (MULTFLT); below is a compact Python workflow to parse, join, and visualize them.

##1.0 Parse MULTFLT (FAULTMULT_AUG-2006.INC)

In [ ]:
from pathlib import Path
import re
import pandas as pd

mult_path = Path(source / "FAULTMULT_AUG-2006.INC")

fault_mult = []
with mult_path.open() as f:
    in_block = False
    for line in f:
        line = line.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("MULTFLT"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        # Example line:  'E_01'  0.01  /
        m = re.match(r"'([^']+)'\s+([0-9.Ee+-]+)", line)
        if m:
            name = m.group(1)
            mult = float(m.group(2))
            fault_mult.append((name, mult))

mult_df = pd.DataFrame(fault_mult, columns=["FAULT", "MULT"])
print(mult_df.head())

      FAULT     MULT
0      E_01  0.01000
1   E_01_F3  0.01000
2      DE_1  3.90000
3  DE_1_LTo  0.01000
4     DE_B3  0.00075


This yields a table of fault names and their transmissibility multipliers.

In [ ]:
# computes a summary of descriptive statistics for a Series or DataFrame.
mult_df.describe()

,MULT
count,60.000000
mean,0.692163
std,2.608318
min,0.000750
25%,0.010000
50%,0.100000
75%,1.000000
max,20.000000


In [ ]:
from skimpy import skim

# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
skim(mult_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 60     │ │ string      │ 1     │                                                          │
│ │ Number of columns │ 2      │ │ float64     │ 1     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column    ┃ NA   ┃ NA %    ┃ mean      ┃ sd       ┃ p0         ┃ p25    ┃ p50   ┃ p75   ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │ MULT      │    0 │       0 │    0.6922 │    2.608 │    0.00075 │   0.01 │   0.1 │     1 │     20 │    █    │  │
│ └───────────┴──────┴─────────┴───────────┴──────────┴────────────┴────────┴───────┴───────┴────────┴─────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓  │
│ ┃ column  ┃ NA  ┃ NA %  ┃ shortest  ┃ longest   ┃ min ┃ max    ┃ chars per row ┃ words per row ┃ total words ┃  │
│ ┡━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩  │
│ │ FAULT   │   0 │     0 │ BC        │ C_08_S_Ti │ BC  │ m_west │          4.63 │             1 │          60 │  │
│ └─────────┴─────┴───────┴───────────┴───────────┴─────┴────────┴───────────────┴───────────────┴─────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [ ]:
# shows the data type of each column in a DataFrame.
mult_df.dtypes

,0
FAULT,object
MULT,float64


In [ ]:
# computes a summary of descriptive statistics for a Series or DataFrame.
mult_df.describe()

,MULT
count,60.000000
mean,0.692163
std,2.608318
min,0.000750
25%,0.010000
50%,0.100000
75%,1.000000
max,20.000000


In [ ]:
# Convert all object columns to category
obj_cols = mult_df.select_dtypes(include="object").columns
mult_df[obj_cols] = mult_df[obj_cols].astype("category")

In [ ]:
# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
skim(mult_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types               Categories                                        │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓ ┏━━━━━━━━━━━━━━━━━━━━━━━┓                                │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃ ┃ Categorical Variables ┃                                │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩ ┡━━━━━━━━━━━━━━━━━━━━━━━┩                                │
│ │ Number of rows    │ 60     │ │ category    │ 1     │ │ FAULT                 │                                │
│ │ Number of columns │ 2      │ │ float64     │ 1     │ └───────────────────────┘                                │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column    ┃ NA   ┃ NA %    ┃ mean      ┃ sd       ┃ p0         ┃ p25    ┃ p50   ┃ p75   ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │ MULT      │    0 │       0 │    0.6922 │    2.608 │    0.00075 │   0.01 │   0.1 │     1 │     20 │    █    │  │
│ └───────────┴──────┴─────────┴───────────┴──────────┴────────────┴────────┴───────┴───────┴────────┴─────────┘  │
│                                                    category                                                     │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                 ┃ NA         ┃ NA %             ┃ ordered                   ┃ unique                ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │ FAULT                  │          0 │                0 │ False                     │                    60 │  │
│ └────────────────────────┴────────────┴──────────────────┴───────────────────────────┴───────────────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

## 2.0 Parse FAULT geometries (FAULT_JUN_05.INC)

In [ ]:
fault_path = Path(source / "FAULT_JUN_05.INC")

cols = ["FAULT", "IX1", "IX2", "IY1", "IY2", "IZ1", "IZ2", "FACE"]
fault_rows = []

with fault_path.open() as f:
    in_block = False
    for line in f:
        raw = line.rstrip("\n")
        line = raw.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("FAULTS"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        # Example:
        # 'G_13'  36   36     65   65      1   22    'X'    /
        parts = raw.split("/")
        if not parts[0].strip():
            continue
        tokens = parts[0].split()
        if len(tokens) < 8:
            continue

        name = tokens[0].strip()
        ix1, ix2, iy1, iy2, iz1, iz2 = map(int, tokens[1:7])
        face = tokens[7].strip()

        # strip quotes around name and face
        name = name.strip("'\"")
        face = face.strip("'\"")

        fault_rows.append((name, ix1, ix2, iy1, iy2, iz1, iz2, face))

fault_df = pd.DataFrame(fault_rows, columns=cols)
print(fault_df.head())

    FAULT  IX1  IX2  IY1  IY2  IZ1  IZ2 FACE
0  m_west    5    5    3    3    1   22    X
1  m_west    5    5    4    4    1   22    X
2  m_west    5    5    5    5    1   22    X
3  m_west    5    5    6    6    1   22    X
4  m_west    5    5    7    7    1   22    X


In [ ]:
# computes a summary of descriptive statistics for a Series or DataFrame.
fault_df.describe()

,IX1,IX2,IY1,IY2,IZ1,IZ2
count,977.000000,977.000000,977.000000,977.000000,977.000000,977.000000
mean,21.512794,21.516888,57.845445,57.845445,2.202661,21.446264
std,10.813153,10.809451,28.407054,28.407054,4.169546,2.057929
min,5.000000,5.000000,2.000000,2.000000,1.000000,10.000000
25%,12.000000,12.000000,37.000000,37.000000,1.000000,22.000000
50%,22.000000,22.000000,59.000000,59.000000,1.000000,22.000000
75%,29.000000,29.000000,81.000000,81.000000,1.000000,22.000000
max,46.000000,46.000000,112.000000,112.000000,20.000000,22.000000


In [ ]:
# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
skim(fault_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 977    │ │ int64       │ 6     │                                                          │
│ │ Number of columns │ 8      │ │ string      │ 2     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column      ┃ NA   ┃ NA %    ┃ mean      ┃ sd        ┃ p0   ┃ p25    ┃ p50    ┃ p75   ┃ p100    ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ IX1         │    0 │       0 │     21.51 │     10.81 │    5 │     12 │     22 │    29 │      46 │  █▇▇█▄▂  │  │
│ │ IX2         │    0 │       0 │     21.52 │     10.81 │    5 │     12 │     22 │    29 │      46 │  █▇▇█▄▂  │  │
│ │ IY1         │    0 │       0 │     57.85 │     28.41 │    2 │     37 │     59 │    81 │     112 │  ▅▅██▆▅  │  │
│ │ IY2         │    0 │       0 │     57.85 │     28.41 │    2 │     37 │     59 │    81 │     112 │  ▅▅██▆▅  │  │
│ │ IZ1         │    0 │       0 │     2.203 │      4.17 │    1 │      1 │      1 │     1 │      20 │    █     │  │
│ │ IZ2         │    0 │       0 │     21.45 │     2.058 │   10 │     22 │     22 │    22 │      22 │       █  │  │
│ └─────────────┴──────┴─────────┴───────────┴───────────┴──────┴────────┴────────┴───────┴─────────┴──────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓  │
│ ┃ column  ┃ NA  ┃ NA %  ┃ shortest  ┃ longest   ┃ min ┃ max    ┃ chars per row ┃ words per row ┃ total words ┃  │
│ ┡━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩  │
│ │ FAULT   │   0 │     0 │ BC        │ C_08_S_Ti │ B2  │ m_west │          4.52 │             1 │         977 │  │
│ │ FACE    │   0 │     0 │ X         │ X         │ X   │ Y      │             1 │             1 │         977 │  │
│ └─────────┴─────┴───────┴───────────┴───────────┴─────┴────────┴───────────────┴───────────────┴─────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [ ]:
# Convert all object columns to category
obj_cols = fault_df.select_dtypes(include="object").columns
fault_df[obj_cols] = fault_df[obj_cols].astype("category")

In [ ]:
skim(fault_df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types               Categories                                        │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓ ┏━━━━━━━━━━━━━━━━━━━━━━━┓                                │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃ ┃ Categorical Variables ┃                                │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩ ┡━━━━━━━━━━━━━━━━━━━━━━━┩                                │
│ │ Number of rows    │ 977    │ │ int64       │ 6     │ │ FAULT                 │                                │
│ │ Number of columns │ 8      │ │ category    │ 2     │ │ FACE                  │                                │
│ └───────────────────┴────────┘ └─────────────┴───────┘ └───────────────────────┘                                │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column      ┃ NA   ┃ NA %    ┃ mean      ┃ sd        ┃ p0   ┃ p25    ┃ p50    ┃ p75   ┃ p100    ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ IX1         │    0 │       0 │     21.51 │     10.81 │    5 │     12 │     22 │    29 │      46 │  █▇▇█▄▂  │  │
│ │ IX2         │    0 │       0 │     21.52 │     10.81 │    5 │     12 │     22 │    29 │      46 │  █▇▇█▄▂  │  │
│ │ IY1         │    0 │       0 │     57.85 │     28.41 │    2 │     37 │     59 │    81 │     112 │  ▅▅██▆▅  │  │
│ │ IY2         │    0 │       0 │     57.85 │     28.41 │    2 │     37 │     59 │    81 │     112 │  ▅▅██▆▅  │  │
│ │ IZ1         │    0 │       0 │     2.203 │      4.17 │    1 │      1 │      1 │     1 │      20 │    █     │  │
│ │ IZ2         │    0 │       0 │     21.45 │     2.058 │   10 │     22 │     22 │    22 │      22 │       █  │  │
│ └─────────────┴──────┴─────────┴───────────┴───────────┴──────┴────────┴────────┴───────┴─────────┴──────────┘  │
│                                                    category                                                     │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                 ┃ NA         ┃ NA %             ┃ ordered                   ┃ unique                ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │ FAULT                  │          0 │                0 │ False                     │                    61 │  │
│ │ FACE                   │          0 │                0 │ False                     │                     2 │  │
│ └────────────────────────┴────────────┴──────────────────┴───────────────────────────┴───────────────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

This gives all fault segments with ijk index ranges and face orientation.

## 3.0 Join geometry and multipliers

In [ ]:
fault_full = fault_df.merge(mult_df, on="FAULT", how="left")
print(fault_full.head())

    FAULT  IX1  IX2  IY1  IY2  IZ1  IZ2 FACE  MULT
0  m_west    5    5    3    3    1   22    X   1.0
1  m_west    5    5    4    4    1   22    X   1.0
2  m_west    5    5    5    5    1   22    X   1.0
3  m_west    5    5    6    6    1   22    X   1.0
4  m_west    5    5    7    7    1   22    X   1.0


In [ ]:
# shows the data type of each column in a DataFrame.
fault_full.dtypes

,0
FAULT,object
IX1,int64
IX2,int64
IY1,int64
IY2,int64
IZ1,int64
IZ2,int64
FACE,category
MULT,float64


Now each segment carries its MULTFLT value, ready for statistics or visualization.

## 4.0 Quick analysis examples

In [ ]:
# Counts and ranges per fault
summary = (
    fault_full
    .groupby("FAULT")
    .agg(
        n_segments=("FAULT", "size"),
        mult=("MULT", "first"),
        ix_min=("IX1", "min"),
        ix_max=("IX2", "max"),
        iy_min=("IY1", "min"),
        iy_max=("IY2", "max"),
        iz_min=("IZ1", "min"),
        iz_max=("IZ2", "max"),
    )
    .reset_index()
)
print(summary.head())

  FAULT  n_segments  mult  ix_min  ix_max  iy_min  iy_max  iz_min  iz_max
0    B2          12   NaN      19      29       6       7       1      22
1    BC          43   0.1       6      46       8      10       1      22
2    CD          13   0.1      11      16      45      52       1      11
3  CD_0           1   1.0       6       7      39      39       1      22
4  CD_1           4   0.1      16      19      53      54       1      22


In [ ]:
# computes a summary of descriptive statistics for a Series or DataFrame.
summary.describe()

,n_segments,mult,ix_min,ix_max,iy_min,iy_max,iz_min,iz_max
count,61.000000,60.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000
mean,16.016393,0.692163,18.754098,22.459016,46.983607,58.311475,3.032787,21.213115
std,20.317884,2.608318,9.686168,10.685775,24.761187,27.509720,5.406685,2.477329
min,1.000000,0.000750,5.000000,5.000000,2.000000,7.000000,1.000000,10.000000
25%,5.000000,0.010000,10.000000,16.000000,30.000000,39.000000,1.000000,22.000000
50%,9.000000,0.100000,19.000000,22.000000,46.000000,52.000000,1.000000,22.000000
75%,18.000000,1.000000,25.000000,29.000000,62.000000,75.000000,1.000000,22.000000
max,115.000000,20.000000,43.000000,46.000000,105.000000,112.000000,20.000000,22.000000


In [ ]:
# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
skim(summary)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 61     │ │ int64       │ 7     │                                                          │
│ │ Number of columns │ 9      │ │ string      │ 1     │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━┳━━━━━┳━━━━━━┳━━━━━━━━┓  │
│ ┃ column       ┃ NA  ┃ NA %                ┃ mean    ┃ sd     ┃ p0       ┃ p25   ┃ p50 ┃ p75 ┃ p100 ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━╇━━━━━╇━━━━━━╇━━━━━━━━┩  │
│ │ n_segments   │   0 │                   0 │   16.02 │  20.32 │        1 │     5 │   9 │  18 │  115 │   █▁   │  │
│ │ mult         │   1 │   1.639344262295082 │  0.6922 │  2.608 │  0.00075 │  0.01 │ 0.1 │   1 │   20 │   █    │  │
│ │ ix_min       │   0 │                   0 │   18.75 │  9.686 │        5 │    10 │  19 │  25 │   43 │ █▅▄▅▃  │  │
│ │ ix_max       │   0 │                   0 │   22.46 │  10.69 │        5 │    16 │  22 │  29 │   46 │ ▆██▃▅▂ │  │
│ │ iy_min       │   0 │                   0 │   46.98 │  24.76 │        2 │    30 │  46 │  62 │  105 │ ▃▃█▄▂▂ │  │
│ │ iy_max       │   0 │                   0 │   58.31 │  27.51 │        7 │    39 │  52 │  75 │  112 │ ▃▄█▄▃▄ │  │
│ │ iz_min       │   0 │                   0 │   3.033 │  5.407 │        1 │     1 │   1 │   1 │   20 │ █    ▁ │  │
│ │ iz_max       │   0 │                   0 │   21.21 │  2.477 │       10 │    22 │  22 │  22 │   22 │      █ │  │
│ └──────────────┴─────┴─────────────────────┴─────────┴────────┴──────────┴───────┴─────┴─────┴──────┴────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓  │
│ ┃ column  ┃ NA  ┃ NA %  ┃ shortest  ┃ longest   ┃ min ┃ max    ┃ chars per row ┃ words per row ┃ total words ┃  │
│ ┡━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩  │
│ │ FAULT   │   0 │     0 │ B2        │ C_08_S_Ti │ B2  │ m_west │          4.59 │             1 │          61 │  │
│ └─────────┴─────┴───────┴───────────┴───────────┴─────┴────────┴───────────────┴───────────────┴─────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

In [ ]:
# Export to CSV
summary.to_csv(dest / "output.csv", index=False)

# Export to Excel
# Single sheet
summary.to_excel(dest / "Summary.xlsx", index=False, freeze_panes=(1,0))

# Multiple sheets in one workbook
#with pd.ExcelWriter(dest + "output.xlsx") as writer:
#    df1.to_excel(writer, sheet_name="Sheet1", index=False)
#    df2.to_excel(writer, sheet_name="Sheet2", index=False)

## 5.0 Plotly visualization on IJ map

Example: show fault traces in IJ space colored by MULTFLT:

In [ ]:
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.express as px
import pandas as pd
from pathlib import Path

traces = []
for name, grp in fault_full.groupby("FAULT"):
    # Represent each segment by its midpoint in I,J and mean K
    i_mid = (grp["IX1"] + grp["IX2"]) / 2
    j_mid = (grp["IY1"] + grp["IY2"]) / 2
    k_mid = (grp["IZ1"] + grp["IZ2"]) / 2
    mult = grp["MULT"].iloc[0] if pd.notna(grp["MULT"].iloc[0]) else 1.0

    traces.append(
        go.Scatter(
            x=i_mid,
            y=j_mid,
            mode="markers",
            marker=dict(
                size=6,
                color=mult,
                colorscale="Viridis",
                showscale=False,   # no colorbar
            ),
            name="",             # no trace name
            text=[f"{name}, k={km}, M={mult}" for km in k_mid],
            hoverinfo="text",
            showlegend=False,    # no legend entry for this trace
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title="Fault segments (IJ midpoints) colored by MULTFLT",
    xaxis_title="I",
    yaxis_title="J",
    showlegend=False,          # hides any legend globally
)

# Enforce equal x and y scales
fig.update_yaxes(
    autorange="reversed",      # Eclipse J increases north->south; flip if desired
    scaleanchor="x",           # lock y scale to x
    constrain="domain"
)
fig.update_xaxes(
    constrain="domain"
)

# Square canvas helps keep aspect
fig.update_layout(width=700, height=700)

# Define title here to be accessible for saving
plot_title = "Fault segments (IJ midpoints) colored by MULTFLT"

# Ensure dest is a Path object for file operations
dest = Path(dest)  # Explicitly convert dest to Path

plotly.offline.plot(fig, filename=str(dest / (plot_title + ".html")))
fig.write_image(str(dest / (plot_title + ".png")), scale=3.125)

fig.show()

This produces an interactive IJ fault map where color encodes transmissibility multiplier, useful for inspecting sealing vs open faults across the grid.

## 6.0 Plotly visualization on IJK map

### 6.1 3D Scatter Plot

In [ ]:
from pathlib import Path
import re
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.express as px

# ----------------------------------------------------------------------
# 1. Parse MULTFLT
# ----------------------------------------------------------------------
mult_path = Path(source / "FAULTMULT_AUG-2006.INC")

fault_mult = []
with mult_path.open() as f:
    in_block = False
    for line in f:
        line = line.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("MULTFLT"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        m = re.match(r"'([^']+)'\s+([0-9.Ee+-]+)", line)
        if m:
            name = m.group(1)
            mult = float(m.group(2))
            fault_mult.append((name, mult))

mult_df = pd.DataFrame(fault_mult, columns=["FAULT", "MULT"])

# ----------------------------------------------------------------------
# 2. Parse FAULTS
# ----------------------------------------------------------------------
fault_path = Path(source / "FAULT_JUN_05.INC")

cols = ["FAULT", "IX1", "IX2", "IY1", "IY2", "IZ1", "IZ2", "FACE"]
fault_rows = []

with fault_path.open() as f:
    in_block = False
    for line in f:
        raw = line.rstrip("\n")
        line = raw.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("FAULTS"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        parts = raw.split("/")
        if not parts[0].strip():
            continue
        tokens = parts[0].split()
        if len(tokens) < 8:
            continue

        name = tokens[0].strip().strip("'\"")
        ix1, ix2, iy1, iy2, iz1, iz2 = map(int, tokens[1:7])
        face = tokens[7].strip().strip("'\"")

        fault_rows.append((name, ix1, ix2, iy1, iy2, iz1, iz2, face))

fault_df = pd.DataFrame(fault_rows, columns=cols)

# ----------------------------------------------------------------------
# 3. Join + compute IJK midpoints
# ----------------------------------------------------------------------
fault_full = fault_df.merge(mult_df, on="FAULT", how="left")
fault_full["MULT"] = fault_full["MULT"].fillna(1.0)

fault_full["I_mid"] = (fault_full["IX1"] + fault_full["IX2"]) / 2.0
fault_full["J_mid"] = (fault_full["IY1"] + fault_full["IY2"]) / 2.0
fault_full["K_mid"] = (fault_full["IZ1"] + fault_full["IZ2"]) / 2.0

# ----------------------------------------------------------------------
# 4. 3D Scatter with per-fault colors and shared MULTFLT colorbar
# ----------------------------------------------------------------------
# Build a qualitative color cycle for different faults
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
]

fault_names = sorted(fault_full["FAULT"].unique())
color_map = {
    name: palette[i % len(palette)]
    for i, name in enumerate(fault_names)
}

traces = []
for i, (name, grp) in enumerate(fault_full.groupby("FAULT")):
    traces.append(
        go.Scatter3d(
            x=grp["I_mid"],
            y=grp["J_mid"],
            z=grp["K_mid"],
            mode="markers",
            name=name,
            marker=dict(
                size=4,
                color=color_map[name],     # legend color by FAULT
                # use MULT as intensity for a shared colorbar
                coloraxis="coloraxis",
            ),
            # hover still shows MULT
            text=[
                f"{name}<br>I={ii:.1f}, J={jj:.1f}, K={kk:.1f}<br>MULT={mm:g}"
                for ii, jj, kk, mm in zip(
                    grp["I_mid"], grp["J_mid"], grp["K_mid"], grp["MULT"]
                )
            ],
            hoverinfo="text",
            showlegend=True,
        )
    )

fig = go.Figure(data=traces)

fig.update_layout(
    title="3D fault segments in IJK (legend by FAULT, colorbar by MULTFLT)",
    scene=dict(
        xaxis_title="I",
        yaxis_title="J",
        zaxis_title="K",
        aspectmode="data",
    ),
    width=900,
    height=750,
    # shared colorbar for MULTFLT using coloraxis
    coloraxis=dict(
        colorscale="Viridis",
        colorbar=dict(title="MULTFLT"),
        cmin=fault_full["MULT"].min(),
        cmax=fault_full["MULT"].max(),
    ),
)

# Define title here to be accessible for saving
plot_title = "3D fault traces in IJK (scatter plot, colored by FAULT)"

# Ensure dest is a Path object for file operations
dest = Path(dest) # Explicitly convert dest to Path

plotly.offline.plot(fig, filename = str(dest / (plot_title + '.html'))) # Changed plotly.offline.plot to pyo.plot and title to plot_title
fig.write_image(str(dest / (plot_title + '.png')), scale=3.125) # Changed title to plot_title

fig.show()

### 3D Line Plot

In [ ]:
from pathlib import Path
import re
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

# ----------------------------------------------------------------------
# 1. Parse MULTFLT
# ----------------------------------------------------------------------
mult_path = Path(source / "FAULTMULT_AUG-2006.INC")

fault_mult = []
with mult_path.open() as f:
    in_block = False
    for line in f:
        line = line.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("MULTFLT"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        m = re.match(r"'([^']+)'\s+([0-9.Ee+-]+)", line)
        if m:
            name = m.group(1)
            mult = float(m.group(2))
            fault_mult.append((name, mult))

mult_df = pd.DataFrame(fault_mult, columns=["FAULT", "MULT"])

# ----------------------------------------------------------------------
# 2. Parse FAULTS
# ----------------------------------------------------------------------
fault_path = Path(source / "FAULT_JUN_05.INC")

cols = ["FAULT", "IX1", "IX2", "IY1", "IY2", "IZ1", "IZ2", "FACE"]
fault_rows = []

with fault_path.open() as f:
    in_block = False
    for line in f:
        raw = line.rstrip("\n")
        line = raw.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("FAULTS"):
            in_block = True
            continue
        if not in_block:
            continue
        if line.startswith("/"):
            break

        parts = raw.split("/")
        if not parts[0].strip():
            continue
        tokens = parts[0].split()
        if len(tokens) < 8:
            continue

        name = tokens[0].strip().strip("'\"")
        ix1, ix2, iy1, iy2, iz1, iz2 = map(int, tokens[1:7])
        face = tokens[7].strip().strip("'\"")

        fault_rows.append((name, ix1, ix2, iy1, iy2, iz1, iz2, face))

fault_df = pd.DataFrame(fault_rows, columns=cols)

# ----------------------------------------------------------------------
# 3. Join + compute IJK midpoints
# ----------------------------------------------------------------------
fault_full = fault_df.merge(mult_df, on="FAULT", how="left")
fault_full["MULT"] = fault_full["MULT"].fillna(1.0)

fault_full["I_mid"] = (fault_full["IX1"] + fault_full["IX2"]) / 2.0
fault_full["J_mid"] = (fault_full["IY1"] + fault_full["IY2"]) / 2.0
fault_full["K_mid"] = (fault_full["IZ1"] + fault_full["IZ2"]) / 2.0

# ----------------------------------------------------------------------
# 4. 3D line plot per fault
# ----------------------------------------------------------------------
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
]
fault_names = sorted(fault_full["FAULT"].unique())
color_map = {name: palette[i % len(palette)] for i, name in enumerate(fault_names)}

traces = []
for name, grp in fault_full.groupby("FAULT"):
    # Sort points to get a smoother line (I, then J, then K)
    grp = grp.sort_values(["I_mid", "J_mid", "K_mid"])

    traces.append(
        go.Scatter3d(
            x=grp["I_mid"],
            y=grp["J_mid"],
            z=grp["K_mid"],
            mode="lines",
            name=name,
            line=dict(
                width=4,
                color=color_map[name],
            ),
            text=[
                f"{name}<br>I={ii:.1f}, J={jj:.1f}, K={kk:.1f}<br>MULT={mm:g}"
                for ii, jj, kk, mm in zip(
                    grp["I_mid"], grp["J_mid"], grp["K_mid"], grp["MULT"]
                )
            ],
            hoverinfo="text",
            showlegend=True,
        )
    )

fig = go.Figure(data=traces)

fig.update_layout(
    title="3D fault traces in IJK (line plot, colored by FAULT)",
    scene=dict(
        xaxis_title="I",
        yaxis_title="J",
        zaxis_title="K",
        aspectmode="data",  # equal I,J,K scale
    ),
    width=900,
    height=750,
)

# Define title here to be accessible for saving
plot_title = "3D fault traces in IJK (line plot, colored by FAULT)"

# Ensure dest is a Path object for file operations
dest = Path(dest) # Explicitly convert dest to Path

plotly.offline.plot(fig, filename = str(dest / (plot_title + '.html'))) # Changed plotly.offline.plot to pyo.plot and title to plot_title
fig.write_image(str(dest / (plot_title + '.png')), scale=3.125) # Changed title to plot_title

fig.show()